# Lesson 4.4 — ADQL Queries in Dash Studio

## Learning Objectives
- Configure ADQL Query data binding in a Dash Studio widget
- Write ADQL queries specifically for the Time Series widget (using `series()`)
- Validate ADQL queries using the green/red indicator in the UI
- Build table widgets and stat widgets backed by ADQL

**Reference**: [ADQL Queries in Dash Studio](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports/dash-studio/data-binding/adql-queries)

## 1. Setting Up ADQL Query Data Binding

### Prerequisites
- Analytics Agent must be deployed and sending data to the Events Service
- The `transactions` (or other) table must have records in the selected time range

### Step-by-step: Add ADQL to a Time Series widget

1. Open your Dash Studio dashboard in **Edit** mode
2. Click **+ Add Widget** → choose **Time Series**
3. In the Data panel on the right:
   - **Show me data for**: Select **ADQL Query** from the dropdown
4. A text area appears — type or paste your ADQL query:
   ```sql
   SELECT series(eventTimestamp, '1m'), userExperience, count(*)
   FROM transactions
   ```
5. **Click outside the text area** to validate
   - ✅ **Green indicator** = valid query, data loading
   - ❌ **Red indicator** = invalid query — error message shown below
6. If valid, the widget updates with data from the current time range
7. Add a title, configure legend, save

> **Important Note from Official Docs**: The `series()` function is **only available for the Time Series widget** in Dash Studio. For table or stat widgets, use aggregation queries without `series()`.

## 2. ADQL Queries for Each Widget Type

Different widget types require different ADQL query shapes:

| Widget Type | ADQL Shape | Example |
|-------------|-----------|----------|
| **Time Series** | `SELECT series(...), [group_field,] aggregate` | `SELECT series(eventTimestamp,'1m'), count(*) FROM transactions` |
| **Table** | `SELECT field1, field2, ... [aggregates] LIMIT n` | `SELECT transactionName, count(*), avg(responseTime) FROM transactions` |
| **Stat / Metric** | `SELECT aggregate FROM table WHERE ...` | `SELECT count(*) FROM transactions WHERE errorOccurred=true` |
| **Pie / Bar** | `SELECT group_field, aggregate FROM table` | `SELECT country, count(*) FROM transactions` |

In [ ]:
# Complete ADQL query library for Dash Studio widgets

ADQL_LIBRARY = {
    # ── TIME SERIES WIDGETS ────────────────────────────────────────────────
    "ts_transaction_count_by_ue": {
        "widget": "Time Series",
        "title": "Transactions per Minute by User Experience",
        "description": "Stacked view of NORMAL/SLOW/VERY_SLOW/ERROR counts over time",
        "query": """SELECT series(eventTimestamp, '1m'), userExperience, count(*)
FROM transactions"""
    },
    "ts_avg_rt_per_bt": {
        "widget": "Time Series",
        "title": "Avg Response Time by Business Transaction",
        "description": "One series per BT — useful for comparing /checkout vs /search",
        "query": """SELECT series(eventTimestamp, '1m'), transactionName, avg(responseTime) AS avgRT
FROM transactions"""
    },
    "ts_error_rate": {
        "widget": "Time Series",
        "title": "Error Count Over Time",
        "description": "Count of errored transactions per minute",
        "query": """SELECT series(eventTimestamp, '1m'), count(*) AS errors
FROM transactions
WHERE errorOccurred = true"""
    },
    "ts_page_load": {
        "widget": "Time Series",
        "title": "Browser Page Load Time Over Time",
        "description": "Average page load time per 5 minutes across all pages",
        "query": """SELECT series(eventTimestamp, '5m'), avg(pageLoadTime) AS avgLoad
FROM browser_records"""
    },
    
    # ── TABLE WIDGETS ──────────────────────────────────────────────────────
    "table_slowest_bts": {
        "widget": "Table",
        "title": "Top 10 Slowest Business Transactions",
        "description": "Rank BTs by maximum response time",
        "query": """SELECT transactionName, count(*) AS total, avg(responseTime) AS avgResponseTime
FROM transactions
ORDER BY avgResponseTime DESC
LIMIT 10"""
    },
    "table_recent_errors": {
        "widget": "Table",
        "title": "Recent Errored Transactions",
        "description": "Latest 20 errored transactions",
        "query": """SELECT transactionName, responseTime, tierName
FROM transactions
WHERE errorOccurred = true
ORDER BY eventTimestamp DESC
LIMIT 20"""
    },
    
    # ── STAT / METRIC WIDGETS ──────────────────────────────────────────────
    "stat_total_errors": {
        "widget": "Stat",
        "title": "Total Errors",
        "description": "Count of all errored transactions in the time window",
        "query": "SELECT count(*) FROM transactions WHERE errorOccurred = true"
    },
    "stat_avg_response_time": {
        "widget": "Stat",
        "title": "Avg Response Time (All BTs)",
        "description": "Overall average response time across all transactions",
        "query": "SELECT avg(responseTime) AS avgRT FROM transactions"
    },
    
    # ── PIE CHART WIDGETS ──────────────────────────────────────────────────
    "pie_errors_by_country": {
        "widget": "Pie Chart",
        "title": "Error Distribution by Country",
        "description": "Geographic breakdown of errors",
        "query": """SELECT country, count(*) AS errors
FROM transactions
WHERE errorOccurred = true
ORDER BY errors DESC"""
    },
}

print("=== Dash Studio ADQL Widget Library ===")
print()
for key, entry in ADQL_LIBRARY.items():
    print(f"[{entry['widget']}] {entry['title']}")
    print(f"  Description: {entry['description']}")
    print(f"  Query:")
    for line in entry['query'].split('\n'):
        print(f"    {line}")
    print()

## 3. Troubleshooting ADQL Validation Errors

When you paste an ADQL query and move the cursor out of the text area, the UI validates it.

### Common errors and fixes:

| Error | Likely Cause | Fix |
|-------|-------------|-----|
| `series() function not supported in this context` | Using `series()` in a non-Time-Series widget | Switch widget type to Time Series |
| `Unknown field 'myField'` | Typo in field name, or field not in the table | Check exact field names in ADQL reference |
| `Table 'xyz' not found` | Wrong table name | Use: `transactions`, `browser_records`, `application_logs` |
| `No data displayed` | Valid query but no records in time range | Widen the time range; verify Analytics Agent is collecting data |
| `Syntax error near 'GROUP'` | ADQL doesn't use explicit GROUP BY | Remove GROUP BY — ADQL groups implicitly |

> **Key Rule from Official Docs**: To view data in the Time Series widget in Dash Studio, you **must** include the `series()` expression in your ADQL query — otherwise the widget will not display a series.

In [ ]:
# EXERCISE 4.4: Fix the broken ADQL queries
#
# Each query below has a bug. Identify the bug and write the corrected query.

broken_queries = [
    {
        "id": 1,
        "widget_type": "Time Series",
        "broken": "SELECT transactionName, count(*) FROM transactions",
        "bug": "???",     # TODO: What's wrong?
        "fixed": "???",   # TODO: Write the corrected query
    },
    {
        "id": 2,
        "widget_type": "Table",
        "broken": "SELECT transactionName, avg(responseTime) FROM transactions GROUP BY transactionName",
        "bug": "???",
        "fixed": "???",
    },
    {
        "id": 3,
        "widget_type": "Table",
        "broken": "SELECT transactionName, count(*) FROM transaction ORDER BY count(*) DESC",
        "bug": "???",   # Hint: look at the table name
        "fixed": "???",
    },
    {
        "id": 4,
        "widget_type": "Time Series",
        "broken": "SELECT series(eventTimestamp, '1m'), userExperience, count(*) FROM transactions WHERE erroroccurred = true",
        "bug": "???",   # Hint: check field name casing
        "fixed": "???",
    },
]

for q in broken_queries:
    print(f"Query {q['id']} [{q['widget_type']}]:")
    print(f"  Broken : {q['broken']}")
    print(f"  Bug    : {q['bug']}")
    print(f"  Fixed  : {q['fixed']}")
    print()

## Summary

- Select **ADQL Query** from the "Show me data for" dropdown in any widget's Data panel
- **Validation** happens automatically when you move the cursor out of the ADQL text area — green = valid, red = error
- The **Time Series widget requires `series()`** in the ADQL query to display a chart — without it, no data is shown
- For Table widgets, use SELECT with aggregates; for Stat widgets, use a single aggregate
- ADQL does NOT support explicit `GROUP BY` — grouping is implicit from SELECT fields

---

**Next**: [Lesson 4.5 — Dashboard Variables](lesson-4.5-dashboard-variables.ipynb)